In [1]:
%cd ../..

/Users/macos/Uni/1st_year/period_2/IntroML/homework


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns

In [3]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})

In [4]:
path_train = "E2/data_E2/penguins_train.csv"
path_test = "E2/data_E2/penguins_test.csv"

In [5]:
df_train = pd.read_csv(path_train)
df_test = pd.read_csv(path_test)

In [6]:
df_train.head()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,species
0,42.3,21.2,191,4150,Adelie
1,41.6,18.0,192,3950,Adelie
2,35.5,17.5,190,3700,Adelie
3,40.2,17.0,176,3450,Adelie
4,39.7,18.4,190,3900,Adelie


In [7]:
df_train['species'].unique()

array(['Adelie', 'notAdelie'], dtype=object)

## Task a

In [8]:
def calc_estimations(df: pd.DataFrame, col: str, specie: str, ddof: int = 2):
    assert col in df

    s = df[df['species'] == specie][col]

    return {
        'mean': s.mean(),
        'std': s.std(ddof=ddof)
    }

def calc_prior(df: pd.DataFrame, pseudocount: int = 1):
    counts = df['species'].value_counts()
    
    prior_Adelie = (counts['Adelie'] + pseudocount) / (len(df) + pseudocount * 1)
    prior_notAdelie = (counts['notAdelie'] + pseudocount) / (len(df) + pseudocount * 2)

    return prior_Adelie, prior_notAdelie


In [9]:
prior_Adelie, prior_notAdelie = calc_prior(df_train)

print(f"p(Y = Adelie)    = {prior_Adelie:.4f}")
print(f"p(Y = notAdelie) = {prior_notAdelie:.4f}")

p(Y = Adelie)    = 0.3421
p(Y = notAdelie) = 0.6623


In [10]:
estimations = []

for specie in ["Adelie", "notAdelie"]:
    for col in ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']:
        out = calc_estimations(df_train, col, specie)

        estimations.append({
            'specie': specie,
            'predictor': col,
            'mean': out['mean'],
            'std': out['std']
        })

df_est = pd.DataFrame.from_records(estimations)

df_est

,specie,predictor,mean,std
0,Adelie,bill_length_mm,38.124,2.841353
1,Adelie,bill_depth_mm,18.336,1.230016
2,Adelie,flipper_length_mm,188.880,6.456005
3,Adelie,body_mass_g,3576.000,471.265641
4,notAdelie,bill_length_mm,47.818,3.636773
5,notAdelie,bill_depth_mm,15.890,1.985809
6,notAdelie,flipper_length_mm,211.300,11.915064
7,notAdelie,body_mass_g,4657.000,795.692178


## Task c

In [11]:
EPS = 1e-6

def calc_accu(ytrue: np.ndarray, ypred: np.ndarray):
    assert ytrue.shape == ypred.shape

    out = 1/len(ytrue) * (np.abs(ytrue - ypred) <= EPS).sum()

    return out

def calc_perplex(ytrue: np.ndarray, ypred_prob: np.ndarray):
    return np.exp(
        -np.sum(
            np.log(ypred_prob[np.arange(len(ytrue)), ytrue])
        ) / len(ytrue)
    )

In [12]:
def pdf_norm(x, predictor: str, target: str):
    r = df_est[(df_est['specie'] == target) & (df_est['predictor'] == predictor)].iloc[0]

    return scipy.stats.norm.pdf(x, r['mean'], r['std'])

In [13]:
df_test.head()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,species
0,39.2,18.6,190,4250,Adelie
1,41.5,18.5,201,4000,Adelie
2,39.0,17.1,191,3050,Adelie
3,37.5,18.9,179,2975,Adelie
4,38.8,17.2,180,3800,Adelie


In [14]:
ytest = (df_test['species'] == 'Adelie').astype(np.int32)

In [15]:
ytest_pred, ytest_pred_prob = [], []

for r in df_test.itertuples():
    p_X_Adelie = pdf_norm(r.bill_length_mm, 'bill_length_mm', 'Adelie') * \
        pdf_norm(r.bill_depth_mm, 'bill_depth_mm', 'Adelie') * \
        pdf_norm(r.flipper_length_mm, 'flipper_length_mm', 'Adelie') * \
        pdf_norm(r.body_mass_g, 'body_mass_g', 'Adelie')

    p_X_notAdelie = pdf_norm(r.bill_length_mm, 'bill_length_mm', 'notAdelie') * \
        pdf_norm(r.bill_depth_mm, 'bill_depth_mm', 'notAdelie') * \
        pdf_norm(r.flipper_length_mm, 'flipper_length_mm', 'notAdelie') * \
        pdf_norm(r.body_mass_g, 'body_mass_g', 'notAdelie')
    
    p_Adelie_X = p_X_Adelie * prior_Adelie / (p_X_Adelie * prior_Adelie + p_X_notAdelie * prior_notAdelie)

    ytest_pred.append(int(p_Adelie_X >= 0.5))
    ytest_pred_prob.append([1 - p_Adelie_X, p_Adelie_X])
    
acc = calc_accu(ytest.to_numpy(), np.array(ytest_pred))
perp = calc_perplex(ytest.to_numpy(), np.array(ytest_pred_prob))

print(f"Accuracy  : {acc:.4f}")
print(f"Perplexity: {perp:.4f}")

Accuracy  : 0.9200
Perplexity: 1.2801


In [17]:
[x[1] for x in ytest_pred_prob[:3]]

[0.9960762725006848, 0.8033489519429122, 0.9986257690579313]